# Afro House Artist Collaboration Network

Builds an artist collaboration network from public Spotify Afro House playlists and finds the most connected artists, bridge artists, and scene communities.

Spotify API credentials are entered at runtime with `getpass`. No keys are stored in this notebook.

In [1]:
!pip -q install spotipy pandas networkx tqdm python-louvain


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.2/354.2 kB 12.1 MB/s eta 0:00:00


In [2]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import getpass

SPOTIPY_CLIENT_ID = getpass.getpass("Enter Spotify Client ID: ")
SPOTIPY_CLIENT_SECRET = getpass.getpass("Enter Spotify Client Secret: ")

sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=SPOTIPY_CLIENT_ID,
        client_secret=SPOTIPY_CLIENT_SECRET
    )
)

Enter Spotify Client ID: ··········
Enter Spotify Client Secret: ··········


In [3]:
sp.search(q="afro house", type="playlist", limit=1)["playlists"]["items"][0]["name"]

'Afro House 2026 🌴 (Top 100) '

In [4]:
from tqdm import tqdm
import pandas as pd

KEYWORDS = [
    "afro house", "afrohouse", "afro-house",
    "afro tech", "afrotech", "deep afro house", "organic house"
]

def search_playlists_multi(keywords, limit_per_kw=20):
    playlists = {}
    for kw in keywords:
        res = sp.search(q=kw, type="playlist", limit=min(limit_per_kw, 50))
        for p in res["playlists"]["items"]:
            if p and p.get("id"):
                playlists[p["id"]] = {
                    "playlist_id": p["id"],
                    "playlist_name": p.get("name"),
                    "owner": (p.get("owner") or {}).get("display_name"),
                    "tracks_total": (p.get("tracks") or {}).get("total"),
                    "keyword": kw
                }
    return pd.DataFrame(list(playlists.values()))

seed_playlists_df = search_playlists_multi(KEYWORDS, limit_per_kw=20)
seed_playlists_df = seed_playlists_df.sort_values("tracks_total", ascending=False).reset_index(drop=True)

seed_playlists_df.head(20), seed_playlists_df.shape

(               playlist_id                                      playlist_name  \
 0   4eKLSEMr4vgNtmHmZuppSh                                  Deep Afro House 🍹   
 1   1ofTRzy58kblfApSvmSCNU                                 Deep House Organic   
 2   0CTLmGbTNu3CSRKtQFuFHY                           Mykonos (Chill & Lounge)   
 3   0nfZioORP3ajjEABjP7SrC                             AFRO DEEP HOUSE | 2024   
 4   0SAo71zNw2vdLZJBMekzhQ                          Afro House/Afro Tech 2025   
 5   6VO19a4U2UKVu5WqWopHZp                     🐦‍🔥 DEEP Tribal/Jungle HOUSE 🪘   
 6   4g4haeURWP6htWZrMXnTSx  Ibiza Lounge 2026🌴 Cocktail, restaurant, & Ibi...   
 7   2JycPQmhwmHFhDUBVTJBJN                     AFRO TECHNO | HOUSE 2026 for U   
 8   5RyZLXT7US07vENKI7AeY2  LATIN HOUSE 2026 🌴  AFRO LATIN  SPANISH TRIBAL...   
 9   5j6C9B81t4HOlM950r10Cw                                      Organic House   
 10  5gt2xlwlnIqEmaNpCA9y7K                                           AFROTECH   
 11  580P0wvHL8e

In [5]:
# Remove obvious non-relevant keywords in titles (quick noise filter)
BAD_TITLE_MARKERS = [
    "workout", "sleep", "study", "focus", "kids", "christian", "gospel",
    "reggaeton", "edm bangers", "hardstyle"
]

def looks_noisy(title):
    t = (title or "").lower()
    return any(b in t for b in BAD_TITLE_MARKERS)

clean_playlists_df = seed_playlists_df[~seed_playlists_df["playlist_name"].apply(looks_noisy)].copy()

clean_playlists_df.head(30), clean_playlists_df.shape

(               playlist_id                                      playlist_name  \
 0   4eKLSEMr4vgNtmHmZuppSh                                  Deep Afro House 🍹   
 1   1ofTRzy58kblfApSvmSCNU                                 Deep House Organic   
 2   0CTLmGbTNu3CSRKtQFuFHY                           Mykonos (Chill & Lounge)   
 3   0nfZioORP3ajjEABjP7SrC                             AFRO DEEP HOUSE | 2024   
 4   0SAo71zNw2vdLZJBMekzhQ                          Afro House/Afro Tech 2025   
 5   6VO19a4U2UKVu5WqWopHZp                     🐦‍🔥 DEEP Tribal/Jungle HOUSE 🪘   
 6   4g4haeURWP6htWZrMXnTSx  Ibiza Lounge 2026🌴 Cocktail, restaurant, & Ibi...   
 7   2JycPQmhwmHFhDUBVTJBJN                     AFRO TECHNO | HOUSE 2026 for U   
 8   5RyZLXT7US07vENKI7AeY2  LATIN HOUSE 2026 🌴  AFRO LATIN  SPANISH TRIBAL...   
 9   5j6C9B81t4HOlM950r10Cw                                      Organic House   
 10  5gt2xlwlnIqEmaNpCA9y7K                                           AFROTECH   
 11  580P0wvHL8e

In [6]:
PILOT_N = 25  # start small
pilot_playlists = clean_playlists_df.head(PILOT_N)["playlist_id"].tolist()

clean_playlists_df.head(PILOT_N)[["playlist_name","owner","tracks_total","keyword"]]

,playlist_name,owner,tracks_total,keyword
0,Deep Afro House 🍹,Bantu Biko,519,deep afro house
1,Deep House Organic,FranckOnTheRoad,479,organic house
2,Mykonos (Chill & Lounge),Dual Sessions,400,organic house
3,AFRO DEEP HOUSE | 2024,K-Music II,374,deep afro house
4,Afro House/Afro Tech 2025,Moon Playlists,362,afrotech
5,🐦‍🔥 DEEP Tribal/Jungle HOUSE 🪘,D.Babz,283,deep afro house
6,"Ibiza Lounge 2026🌴 Cocktail, restaurant, & Ibi...",Shattered Nights,279,deep afro house
7,AFRO TECHNO | HOUSE 2026 for U,For U Music,276,afro tech
8,LATIN HOUSE 2026 🌴 AFRO LATIN SPANISH TRIBAL...,LATIN DEEJAYS,270,deep afro house
9,Organic House,@chrisluno,268,organic house


In [7]:
def get_all_playlist_items(playlist_id, max_items=2000):
    items = []
    offset = 0
    while True:
        batch = sp.playlist_items(
            playlist_id,
            offset=offset,
            limit=100,
            additional_types=("track",)
        )
        items.extend(batch["items"])
        offset += len(batch["items"])
        if batch["next"] is None or len(items) >= max_items:
            break
    return items[:max_items]

tracks_rows = []

for pid in tqdm(pilot_playlists, total=len(pilot_playlists)):
    meta = clean_playlists_df.loc[clean_playlists_df["playlist_id"] == pid].iloc[0].to_dict()
    items = get_all_playlist_items(pid, max_items=1500)
    for it in items:
        tr = (it.get("track") or {})
        if not tr or not tr.get("id"):
            continue
        artists = tr.get("artists") or []
        tracks_rows.append({
            "playlist_id": pid,
            "playlist_name": meta["playlist_name"],
            "track_id": tr["id"],
            "track_name": tr.get("name"),
            "track_popularity": tr.get("popularity"),
            "album_release_date": (tr.get("album") or {}).get("release_date"),
            "artist_ids": [a["id"] for a in artists if a.get("id")],
            "artist_names": [a["name"] for a in artists if a.get("name")]
        })

tracks_df = pd.DataFrame(tracks_rows).drop_duplicates(subset=["playlist_id","track_id"]).reset_index(drop=True)
tracks_df.head(), tracks_df.shape

100%|██████████| 25/25 [00:45<00:00,  1.81s/it]


(              playlist_id      playlist_name                track_id  \
 0  4eKLSEMr4vgNtmHmZuppSh  Deep Afro House 🍹  4phq223wnI807LaLlr3LYM   
 1  4eKLSEMr4vgNtmHmZuppSh  Deep Afro House 🍹  3QRLD0W4JOj9W2DdaQ0cAd   
 2  4eKLSEMr4vgNtmHmZuppSh  Deep Afro House 🍹  0ZwLxloVnLKoG3wX0q3hGf   
 3  4eKLSEMr4vgNtmHmZuppSh  Deep Afro House 🍹  2WgPmBVhQDXidR8oQhS8J0   
 4  4eKLSEMr4vgNtmHmZuppSh  Deep Afro House 🍹  2ULz7EuDpQ9iifrbHLmpbx   
 
                                   track_name  track_popularity  \
 0  Colours of Love - The Layabouts Vocal Mix                 0   
 1                               Perfect Girl                29   
 2                         Salanabani - Remix                29   
 3                                 City Rains                 0   
 4                                    J'adore                53   
 
   album_release_date                                         artist_ids  \
 0         2013-11-18   [6XRpgVJmeNjNh04Pjq07Wm, 4Gfgfu8VYfDlejKCKU9mpc]   
 1  

In [8]:
# How many unique tracks and artists did we get?
n_tracks = tracks_df["track_id"].nunique()
n_artists = len({aid for ids in tracks_df["artist_ids"] for aid in ids})

n_tracks, n_artists

(5458, 4724)

In [9]:
import pandas as pd
import numpy as np

# --- 1. Explode artists (one artist per row) ---
artist_tracks = (
    tracks_df
    .explode(["artist_ids", "artist_names"])
    .rename(columns={
        "artist_ids": "artist_id",
        "artist_names": "artist_name"
    })
)

# --- 2. Clean nulls ---
artist_tracks = artist_tracks[
    artist_tracks["artist_id"].notna() &
    artist_tracks["artist_name"].notna()
]

# --- 3. Ensure popularity is numeric ---
artist_tracks["track_popularity"] = pd.to_numeric(
    artist_tracks["track_popularity"],
    errors="coerce"
)

# --- 4. Aggregate to artist level ---
artist_spotify_df = (
    artist_tracks
    .groupby(["artist_id", "artist_name"], as_index=False)
    .agg(
        avg_track_popularity=("track_popularity", "mean"),
        max_track_popularity=("track_popularity", "max"),
        num_tracks_in_afrohouse_playlists=("track_id", "nunique")
    )
)

# --- 5. Optional: add country placeholder (merge later if needed) ---
artist_spotify_df["country"] = "Unknown"

# --- 6. Round for readability ---
artist_spotify_df["avg_track_popularity"] = artist_spotify_df["avg_track_popularity"].round(2)
artist_spotify_df["max_track_popularity"] = artist_spotify_df["max_track_popularity"].round(2)

artist_spotify_df.head(), artist_spotify_df.shape

(                artist_id  artist_name  avg_track_popularity  \
 0  003ScA1EGXbCLpKFucHXb3    Eli Kazah                 29.00   
 1  004F88I7EWH54o9jNfUbEn    Bellestar                 46.33   
 2  007AIeFEFncgi8Lbvn6XC0  Beard2Beard                 21.00   
 3  009CRdhmFYn8IdxNRsemvV      Zikhona                  8.00   
 4  00CJAni0JgRC6EhKTKUcCG        mcfly                 39.00   
 
    max_track_popularity  num_tracks_in_afrohouse_playlists  country  
 0                    29                                  1  Unknown  
 1                    55                                 21  Unknown  
 2                    23                                  2  Unknown  
 3                     8                                  1  Unknown  
 4                    39                                  1  Unknown  ,
 (4724, 6))

In [10]:
import time

def chunked(lst, n=50):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

all_artist_ids = sorted({aid for ids in tracks_df["artist_ids"] for aid in ids})
len(all_artist_ids)

4724

In [11]:
artist_rows = []

for chunk in chunked(all_artist_ids, 50):
    data = sp.artists(chunk)["artists"]
    for a in data:
        if not a:
            continue
        artist_rows.append({
            "artist_id": a["id"],
            "artist_name": a["name"],
            "genres": a.get("genres", []),
            "followers": (a.get("followers") or {}).get("total"),
            "artist_popularity": a.get("popularity")
        })
    time.sleep(0.05)

artists_df = pd.DataFrame(artist_rows).drop_duplicates("artist_id")
artists_df.head(), artists_df.shape

(                artist_id  artist_name                              genres  \
 0  003ScA1EGXbCLpKFucHXb3    Eli Kazah                       [lo-fi beats]   
 1  004F88I7EWH54o9jNfUbEn    Bellestar                [lounge, bossa nova]   
 2  007AIeFEFncgi8Lbvn6XC0  Beard2Beard                     [organic house]   
 3  009CRdhmFYn8IdxNRsemvV      Zikhona  [afro tech, afro house, afro soul]   
 4  00CJAni0JgRC6EhKTKUcCG        mcfly             [afro house, afro tech]   
 
    followers  artist_popularity  
 0       1053                 24  
 1      13816                 57  
 2       3471                 24  
 3        234                  3  
 4        912                 27  ,
 (4724, 5))

In [12]:
GENRE_MARKERS = [
    "afro house", "afro-house", "afrohouse",
    "afro tech", "afrotech",
    "deep afro", "organic house",
    "amapiano"
]

def is_afro_relevant(genres):
    g = " | ".join([x.lower() for x in (genres or [])])
    return any(m in g for m in GENRE_MARKERS)

artists_df["afro_relevant"] = artists_df["genres"].apply(is_afro_relevant)
artists_df["afro_relevant"].value_counts()


,count
afro_relevant,
True,2372
False,2352


In [13]:
afro_artist_ids = set(artists_df.loc[artists_df["afro_relevant"], "artist_id"])

tracks_df["afro_relevant_track"] = tracks_df["artist_ids"].apply(
    lambda ids: any(a in afro_artist_ids for a in ids)
)

afro_tracks_df = tracks_df[tracks_df["afro_relevant_track"]].copy()

afro_tracks_df.shape

(4396, 9)

In [14]:
afro_tracks_df.sample(20, random_state=1)[["track_name","playlist_name","artist_names"]]

,track_name,playlist_name,artist_names
4954,Famax - Edit,TRIBAL HOUSE 2026 🥁 AFRO HOUSE 2026,[RAFFA GUIDO]
5349,Se te nota,AFRO HOUSE - MALANDREO 2026 💊🤪🍭,[Yiyiflirt]
1666,Tekahela,AFRO DEEP HOUSE | 2024,"[Dj Coach, Dj Vitoto, Ray T]"
4030,I Adore You (feat. Daecolm),Beach Music 🌴 Chill Summer Vibes,"[HUGEL, Topic, Arash, Daecolm]"
797,Minero - M.RUX Remix,Deep House Organic,"[Rodrigo Gallardo, M.RUX]"
3604,Zid Lmel - FNX Omar Remix,AFROTECH,"[Boddhi Satva, Maalem Hammam, FNX Omar]"
2813,Muyè - Black Coffee Remix,AFRO TECHNO | HOUSE 2026 for U,"[Rampa, &ME, Adam Port, Black Coffee]"
3848,Doesn’t Matter,AFRO HOUSE MIX 2026 🔥 Party Afro House Remixes,"[DRULZ, BOOTY LEAK, Afro Queen]"
731,Sinere - Elfenberg Remix,Deep House Organic,"[rBert, Elfenberg]"
129,Lost (with DJ Angelo feat. Jinadu),Deep Afro House 🍹,"[Black Coffee, Angelos, Jinadu]"


In [15]:
from collections import defaultdict
import itertools

edge_weight = defaultdict(int)
edge_examples = defaultdict(list)

for _, row in afro_tracks_df.iterrows():
    artist_ids = list(dict.fromkeys(row["artist_ids"]))  # unique
    if len(artist_ids) < 2:
        continue

    for a, b in itertools.combinations(sorted(artist_ids), 2):
        edge_weight[(a, b)] += 1
        if len(edge_examples[(a, b)]) < 3:
            edge_examples[(a, b)].append(row["track_name"])

edges = []
for (a, b), w in edge_weight.items():
    edges.append({
        "artist_id_1": a,
        "artist_id_2": b,
        "weight": w,
        "example_tracks": "; ".join(edge_examples[(a, b)])
    })

edges_df = pd.DataFrame(edges).sort_values("weight", ascending=False).reset_index(drop=True)

edges_df.head(20), edges_df.shape

(               artist_id_1             artist_id_2  weight  \
 0   26WKgv73kRHD0gEDKD1i8j  2loEsOijJ6XiGzWYFXMIRk      37   
 1   08jywfUS0hp8XYlYs0cvz8  5mIowAJMp7RKNheelruV5z      35   
 2   08jywfUS0hp8XYlYs0cvz8  26WKgv73kRHD0gEDKD1i8j      35   
 3   08jywfUS0hp8XYlYs0cvz8  2loEsOijJ6XiGzWYFXMIRk      31   
 4   2loEsOijJ6XiGzWYFXMIRk  5mIowAJMp7RKNheelruV5z      31   
 5   26WKgv73kRHD0gEDKD1i8j  5mIowAJMp7RKNheelruV5z      29   
 6   2IDtMW47SEAptw9RwNREm0  6TILJrqby5UzMV1EemkxtN      21   
 7   2loEsOijJ6XiGzWYFXMIRk  6XjgqvaVLUib6lCerbZrYD      19   
 8   37lTKHi901VnwngQ7C8noT  6XjgqvaVLUib6lCerbZrYD      19   
 9   2loEsOijJ6XiGzWYFXMIRk  37lTKHi901VnwngQ7C8noT      19   
 10  2sSSGlRMfz4ZEcw4rw0m0v  78h7VfOuRAnd6jtaEYQpg3      18   
 11  44qAhQu52dYKcHOFQd3esf  6lbUCWVW3hgQgrJwB8wadJ      15   
 12  0nyKVqkATGrYXA98ieTJr0  2sSSGlRMfz4ZEcw4rw0m0v      15   
 13  1IFAU4mznUcfPVP9z2c24N  5PlfkPxwCpRRWQJBxCa0By      14   
 14  0iQaClnw8ysyL24QKIeFuT  46EZcT2VCmBxSG5xdmNbix    

**Due Diligence Checks**


In [16]:
# Check all edge artist IDs exist in artists_df
edge_artist_ids = set(edges_df["artist_id_1"]).union(set(edges_df["artist_id_2"]))
artist_ids = set(artists_df["artist_id"])

missing_ids = edge_artist_ids - artist_ids
len(missing_ids)

0

In [17]:
#Self-Loops Check#
self_loops = edges_df[edges_df["artist_id_1"] == edges_df["artist_id_2"]]
self_loops.shape

(0, 4)

In [18]:
#Edge Weight Distribution Check#
edges_df["weight"].describe()

,weight
count,4848.000000
mean,1.585396
std,1.852440
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,37.000000


In [19]:
import networkx as nx

G_tmp = nx.Graph()
G_tmp.add_weighted_edges_from(
    [(r["artist_id_1"], r["artist_id_2"], r["weight"]) for _, r in edges_df.iterrows()]
)

components = [len(c) for c in nx.connected_components(G_tmp)]
max(components), len(components)

(2230, 308)

In [20]:
edges_df_final = edges_df.copy()
artists_df_final = artists_df.copy()

**Build Network Graph**

In [21]:
import networkx as nx

G = nx.Graph()

for _, r in edges_df.iterrows():
    G.add_edge(r["artist_id_1"], r["artist_id_2"], weight=int(r["weight"]))

G.number_of_nodes(), G.number_of_edges()

(3085, 4848)

In [22]:
import pandas as pd
import community.community_louvain as community_louvain # Corrected import
import networkx as nx

# Weighted degree
degree_w = dict(G.degree(weight="weight"))

# Betweenness (weighted)
betweenness = nx.betweenness_centrality(G, weight="weight", normalized=True)

# Louvain communities
partition = community_louvain.best_partition(G, weight="weight")

# Eigenvector (weighted) for the largest connected component
# The graph G is disconnected, which causes AmbiguousSolution error for the whole graph.
# We will compute eigenvector centrality for the largest connected component only.
largest_cc = max(nx.connected_components(G), key=len)
G_largest_cc = G.subgraph(largest_cc)

# Check if the largest connected component has at least one node to avoid errors
if len(G_largest_cc) > 0:
    eigenvector_cc = nx.eigenvector_centrality_numpy(G_largest_cc, weight="weight")
    # Map the results back to all nodes, setting 0 for nodes not in the largest component
    eigenvector = {node: eigenvector_cc.get(node, 0.0) for node in G.nodes()}
else:
    # If for some reason the largest component is empty, set all to 0
    eigenvector = {node: 0.0 for node in G.nodes()}

metrics_df = pd.DataFrame({
    "artist_id": list(G.nodes()),
    "degree_weighted": [degree_w.get(n, 0) for n in G.nodes()],
    "betweenness": [betweenness.get(n, 0.0) for n in G.nodes()],
    "eigenvector": [eigenvector.get(n, 0.0) for n in G.nodes()],
    "community_id": [partition.get(n, -1) for n in G.nodes()]
})

# Join artist metadata (name, followers, popularity, genres)
metrics_df = metrics_df.merge(
    artists_df[["artist_id","artist_name","followers","artist_popularity","genres","afro_relevant"]],
    on="artist_id",
    how="left"
)

metrics_df.sort_values("degree_weighted", ascending=False).head(20)

,artist_id,degree_weighted,betweenness,eigenvector,community_id,artist_name,followers,artist_popularity,genres,afro_relevant
8,2sSSGlRMfz4ZEcw4rw0m0v,220,0.112725,2.215031e-06,2,Cafe De Anatolia,138878,56,"[organic house, afro house, deep house, tribal...",True
15,0iQaClnw8ysyL24QKIeFuT,219,0.054784,1.047524e-09,5,Afro Queen,3065,61,[afro house],True
1,2loEsOijJ6XiGzWYFXMIRk,210,0.004713,4.913863e-01,0,Adam Port,353687,73,"[afro house, house, afro tech]",True
0,26WKgv73kRHD0gEDKD1i8j,198,0.004147,4.861261e-01,0,Keinemusik,563677,73,"[afro house, melodic house, house, afro tech]",True
14,5PlfkPxwCpRRWQJBxCa0By,176,0.045741,1.818507e-04,4,HUGEL,741884,80,"[latin house, afro house, techengue]",True
2,08jywfUS0hp8XYlYs0cvz8,168,0.006181,4.447538e-01,0,Rampa,233191,67,"[afro house, afro tech, melodic house, house]",True
3,5mIowAJMp7RKNheelruV5z,145,0.002044,4.185297e-01,0,&ME,235736,66,"[afro house, afro tech, house, melodic house]",True
58,0aPNhtrzvas6BBr9N7Db7w,139,0.049368,1.402246e-07,29,Sounds Of Sirin,2502,44,[organic house],True
140,6wMr4zKPrrR0UVz08WtUWc,99,0.030835,5.587880e-02,10,Black Coffee,1547537,67,"[afro house, afro tech, house]",True
10,44qAhQu52dYKcHOFQd3esf,91,0.023468,1.134576e-04,3,Francis Mercier,124252,66,"[afro house, afro tech, melodic house, tribal ...",True


In [23]:
metrics_df.shape, metrics_df["community_id"].nunique(), metrics_df["afro_relevant"].value_counts(dropna=False)

# Identify largest community sizes
metrics_df["community_id"].value_counts().head(10)

,count
community_id,
2,185
4,149
10,144
12,140
5,126
3,111
29,106
22,105
19,96


In [24]:
import os
os.makedirs("outputs", exist_ok=True)

metrics_df.to_csv("outputs/artist_metrics.csv", index=False)
edges_df.to_csv("outputs/edges_collab.csv", index=False)
afro_tracks_df.to_csv("outputs/tracks_afro_filtered.csv", index=False)
artists_df.to_csv("outputs/artists_all.csv", index=False)

print("Saved: outputs/artist_metrics.csv, edges_collab.csv, tracks_afro_filtered.csv, artists_all.csv")

Saved: outputs/artist_metrics.csv, edges_collab.csv, tracks_afro_filtered.csv, artists_all.csv


In [25]:
import networkx as nx
import numpy as np

num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

components = list(nx.connected_components(G))
component_sizes = sorted([len(c) for c in components], reverse=True)
largest_cc_size = component_sizes[0]
largest_cc_share = largest_cc_size / num_nodes

density = nx.density(G)

num_nodes, num_edges, len(components), largest_cc_size, largest_cc_share, density

(3085, 4848, 308, 2230, 0.7228525121555915, 0.0010191147071621817)

In [26]:
top_bridges = metrics_df.sort_values("betweenness", ascending=False).head(20)
top_bridges[["artist_name","betweenness","degree_weighted","community_id","followers","artist_popularity"]]

,artist_name,betweenness,degree_weighted,community_id,followers,artist_popularity
62,Idd Aziz,0.115567,80,3,21467,52
8,Cafe De Anatolia,0.112725,220,2,138878,56
15,Afro Queen,0.054784,219,5,3065,61
299,Shimza,0.050290,42,12,230409,63
58,Sounds Of Sirin,0.049368,139,29,2502,44
185,Da Capo,0.049330,80,10,141018,44
302,Toshi,0.048872,21,19,7846,40
14,HUGEL,0.045741,176,4,741884,80
106,Stevo Atambire,0.040994,19,23,9092,51
836,Aaron Sevilla,0.038300,22,4,65787,53


In [27]:
t1 = metrics_df.sort_values("degree_weighted", ascending=False).head(20)
t1 = t1[["artist_name","degree_weighted","betweenness","eigenvector","followers","artist_popularity","community_id"]]
t1

,artist_name,degree_weighted,betweenness,eigenvector,followers,artist_popularity,community_id
8,Cafe De Anatolia,220,0.112725,2.215031e-06,138878,56,2
15,Afro Queen,219,0.054784,1.047524e-09,3065,61,5
1,Adam Port,210,0.004713,4.913863e-01,353687,73,0
0,Keinemusik,198,0.004147,4.861261e-01,563677,73,0
14,HUGEL,176,0.045741,1.818507e-04,741884,80,4
2,Rampa,168,0.006181,4.447538e-01,233191,67,0
3,&ME,145,0.002044,4.185297e-01,235736,66,0
58,Sounds Of Sirin,139,0.049368,1.402246e-07,2502,44,29
140,Black Coffee,99,0.030835,5.587880e-02,1547537,67,10
10,Francis Mercier,91,0.023468,1.134576e-04,124252,66,3


In [28]:
t2 = metrics_df.sort_values("betweenness", ascending=False).head(20)
t2 = t2[["artist_name","betweenness","degree_weighted","followers","artist_popularity","community_id"]]
t2

,artist_name,betweenness,degree_weighted,followers,artist_popularity,community_id
62,Idd Aziz,0.115567,80,21467,52,3
8,Cafe De Anatolia,0.112725,220,138878,56,2
15,Afro Queen,0.054784,219,3065,61,5
299,Shimza,0.050290,42,230409,63,12
58,Sounds Of Sirin,0.049368,139,2502,44,29
185,Da Capo,0.049330,80,141018,44,10
302,Toshi,0.048872,21,7846,40,19
14,HUGEL,0.045741,176,741884,80,4
106,Stevo Atambire,0.040994,19,9092,51,23
836,Aaron Sevilla,0.038300,22,65787,53,4


In [29]:
comm_sizes = metrics_df["community_id"].value_counts().reset_index()
comm_sizes.columns = ["community_id","size"]

def top_artists_in_comm(cid, n=5):
    df = metrics_df[metrics_df["community_id"] == cid].sort_values("degree_weighted", ascending=False).head(n)
    return ", ".join(df["artist_name"].fillna("Unknown").tolist())

comm_sizes["top_artists"] = comm_sizes["community_id"].apply(lambda cid: top_artists_in_comm(cid, 5))
comm_sizes.head(10)

,community_id,size,top_artists
0,2,185,"Cafe De Anatolia, Rialians on Earth, Billy Est..."
1,4,149,"HUGEL, Daecolm, Jesús Fernández, Topic, Arash"
2,10,144,"Black Coffee, Da Capo, Sun-El Musician, Zakes ..."
3,12,140,"Black Motion, Shimza, DJ Maphorisa, Kabza De S..."
4,5,126,"Afro Queen, Notscandal, Make it Maks, beyondli..."
5,3,111,"Francis Mercier, Idd Aziz, Nitefreak, FNX OMAR..."
6,29,106,"Sounds Of Sirin, DJ Phellix, Seventh Soul, Oli..."
7,22,105,"Djuma Soundsystem, Dorian Craft, Nandu, Oluhle..."
8,19,96,"Lizwi, Kususa, Toshi, Armonica, Citizen Deep"
9,1,91,"Maz, Samm (BE), Antdot, Tabia, Ajna (BE)"


In [30]:
nodes_tableau = metrics_df[[
    "artist_id","artist_name","degree_weighted","betweenness","eigenvector",
    "community_id","followers","artist_popularity","afro_relevant"
]].copy()

edges_tableau = edges_df[["artist_id_1","artist_id_2","weight"]].copy()

nodes_tableau.to_csv("outputs/nodes_tableau.csv", index=False)
edges_tableau.to_csv("outputs/edges_tableau.csv", index=False)

print("Saved: outputs/nodes_tableau.csv and outputs/edges_tableau.csv")

Saved: outputs/nodes_tableau.csv and outputs/edges_tableau.csv


In [31]:
afro_artist_ids = set(artists_df.loc[artists_df["afro_relevant"], "artist_id"])

edges_df_strict = edges_df[
    edges_df["artist_id_1"].isin(afro_artist_ids) &
    edges_df["artist_id_2"].isin(afro_artist_ids)
].copy()

edges_df.shape, edges_df_strict.shape

((4848, 4), (2907, 4))

In [32]:
nodes_tableau = metrics_df[[
    "artist_id",
    "artist_name",
    "degree_weighted",
    "betweenness",
    "eigenvector",
    "community_id",
    "followers",
    "artist_popularity",
    "afro_relevant"
]].copy()

nodes_tableau.shape

(3085, 9)

In [33]:
edges_tableau = edges_df[[
    "artist_id_1",
    "artist_id_2",
    "weight"
]].copy()

edges_tableau.shape

(4848, 3)

In [34]:
edges_df_gephi = edges_df.rename(columns={
    "artist_id_1": "Source",
    "artist_id_2": "Target",
    "weight": "Weight"
}).copy()

# Required for Gephi clarity
edges_df_gephi["Type"] = "Undirected"

# Clean + enforce types
edges_df_gephi["Source"] = edges_df_gephi["Source"].astype(str).str.strip()
edges_df_gephi["Target"] = edges_df_gephi["Target"].astype(str).str.strip()
edges_df_gephi["Weight"] = pd.to_numeric(edges_df_gephi["Weight"], errors="coerce").fillna(1)

# Remove self-loops (important for clean network metrics)
edges_df_gephi = edges_df_gephi[
    edges_df_gephi["Source"] != edges_df_gephi["Target"]
]

edges_df_gephi.to_csv("edges_gephi.csv", index=False)

In [35]:
nodes_df_gephi = nodes_tableau.rename(columns={
    "artist_id": "Id",
    "artist_name": "Label"
}).copy()

nodes_df_gephi["Id"] = nodes_df_gephi["Id"].astype(str).str.strip()
nodes_df_gephi["Label"] = nodes_df_gephi["Label"].astype(str).str.strip()

nodes_df_gephi.to_csv("nodes_gephi.csv", index=False)

In [36]:
nodes_tableau.to_csv("nodes_tableau.csv", index=False)
edges_tableau.to_csv("edges_tableau.csv", index=False)

print("Saved: nodes_tableau.csv and edges_tableau.csv")

Saved: nodes_tableau.csv and edges_tableau.csv


In [37]:
artist_spotify_df.to_csv(
    "spotify_afrohouse_artists_tableau.csv",
    index=False
)